# [M4] AI NHÌN VÀ HIỂU DỮ LIỆU
## Buổi S06: Biểu Diễn Văn Bản: Tokenization & Xây Dựng Từ Điển

---

### Metadata Khóa Học
* **Module:** M4 — AI NHÌN VÀ HIỂU DỮ LIỆU
* **Buổi:** S06
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook xây dựng hệ thống Tokenization và Vocabulary chuẩn NLP
* **Mức độ liên quan VOAI / Olympic AI:** Nền móng xử lý văn bản cho mọi bài toán NLP trong các kỳ thi Olympic AI.

#### Mục tiêu học tập (Learning Objectives)
- Hiểu thách thức căn bản của Xử lý Ngôn ngữ Tự nhiên (NLP): Máy tính chỉ hiểu số học, không hiểu chuỗi ký tự rời rạc.
- Xây dựng pipeline Tiền xử lý văn bản chuẩn: Hạ chữ thường (Lowercase), khử ký tự đặc biệt, chuẩn hóa khoảng trắng bằng Regular Expressions (Regex).
- Nắm vững các cấp độ Tokenization: Tách từ (Word-level), Tách ký tự (Character-level), và trực giác về Subword BPE.
- Tự tay lập trình cấu trúc Từ Điển Từ Vựng (Vocabulary): Mã hóa từ sang chỉ số (Word-to-Index) và xử lý từ mới chưa từng thấy (Out-Of-Vocabulary - `<UNK>`).

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành các buổi trước của M4 (Hiểu tư duy xử lý dữ liệu phi cấu trúc).
- Kỹ năng xử lý chuỗi cơ bản trong Python (chuỗi `str`, phương thức `.split()`).

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Trong Computer Vision, một bức ảnh bản chất đã là các con số cường độ sáng từ 0 đến 255. Nhưng trong Ngôn ngữ Tự nhiên (NLP), một câu như: *"Đề thi Olympic năm nay rất hay và thách thức!"* chỉ là một mảng các ký tự Unicode vô nghĩa đối với các thuật toán tối ưu hóa ma trận.

Làm thế nào để biến những câu chữ giàu cảm xúc, đầy tính ngữ cảnh và ngữ pháp của con người thành những con số toán học mà máy tính có thể hiểu được? **Chuẩn hóa văn bản (Preprocessing), Phân tách từ (Tokenization) và Xây dựng Từ điển (Vocabulary Mapping)** chính là cây cầu đầu tiên kết nối giữa ngôn ngữ loài người và trí tuệ nhân tạo.



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Chu trình tiền xử lý văn bản thô (Text Preprocessing)
1. **Chuyển về chữ thường (Lowercasing):** Giúp đồng nhất `'Olympic'` và `'olympic'` thành cùng một khái niệm từ vựng.
2. **Loại bỏ nhiễu bằng Regex:** Xóa bỏ mã HTML (`<p>`, `<div>`), ký tự đặc biệt thừa (`@`, `#`, `$`), và các dấu chấm câu nếu không cần thiết.
3. **Chuẩn hóa khoảng trắng:** Loại bỏ dấu cách thừa, ký tự xuống dòng `\n`, `\t`.

### 2. Các cấp độ Tokenization
* **Word-level Tokenization (Tách theo từ):** Đơn giản, trực quan. Nhược điểm: Kích thước từ điển khổng lồ, dễ bị lỗi Out-Of-Vocabulary (OOV) khi gặp từ mới hoặc từ viết sai chính tả.
* **Character-level Tokenization (Tách theo ký tự):** Từ điển rất nhỏ (chỉ gồm bảng chữ cái), không bao giờ bị OOV. Nhược điểm: Chuỗi trở nên quá dài, mất đi ý nghĩa trọn vẹn của từng từ.
* **Subword Tokenization (BPE / WordPiece):** Cân bằng hoàn hảo giữa hai phương pháp trên (nền tảng của BERT, GPT).

### 3. Cấu trúc Từ Điển (Vocabulary Indexer) & Special Tokens
Một từ điển chuẩn mực cần các token đặc biệt:
* `<PAD>` (Index 0): Dùng để đệm các câu ngắn cho bằng độ dài tối đa của batch.
* `<UNK>` (Index 1): Đại diện cho các từ lạ chưa từng xuất hiện trong tập huấn luyện (Out-of-Vocabulary).
* `<SOS>` / `<EOS>`: Đánh dấu điểm bắt đầu (Start of Sentence) và kết thúc (End of Sentence).



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Quy trình biến đổi Văn bản thành Chỉ số Token:
"Olympic AI rất hay!"
        │
        ▼ (Tiền xử lý Regex & Lowercase)
"olympic ai rat hay"
        │
        ▼ (Tokenization)
['olympic', 'ai', 'rat', 'hay']
        │
        ▼ (Tra cứu Từ điển Vocabulary)
Token IDs: [ 45, 12, 108, 89 ]  ──> Sẵn sàng chuyển thành PyTorch Tensor!
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import re

# 1. Pipeline tien xu ly van ban bang Regular Expressions (Regex)
def clean_text(text):
    # Ha chu thuong
    text = text.lower()
    # Loai bo ma HTML neu co
    text = re.sub(r'<.*?>', '', text)
    # Loai bo cac ky tu dac biet, chi giu lai chu cai, so va dau cach
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    # Xoa khoang trang thua
    text = re.sub(r'\s+', ' ', text).strip()
    return text

sample_raw = "  <p>Chao mung toi VOAI 2026! De thi co 38 sessions: Rat tuyet voi??? </p> "
cleaned = clean_text(sample_raw)
print("Van ban goc :", sample_raw)
print("Van ban sach:", cleaned)

### 🔍 Code Walkthrough (Khối 1)
Khối code minh họa hàm làm sạch chuỗi chuẩn mực bằng Regex:
- Loại bỏ thẻ HTML, dấu câu thừa và chuẩn hóa khoảng trắng.
- Đưa văn bản về dạng chữ thường sạch sẽ, sẵn sàng cho bước phân tách từ.


In [ ]:
# 2. Xay dung lop Vocabulary Indexer tu so 0
class SimpleVocabulary:
    def __init__(self, min_freq=1):
        self.min_freq = min_freq
        self.pad_token = "<PAD>"
        self.unk_token = "<UNK>"
        
        # Khoi tao bang tra cuu 2 chieu
        self.word2idx = {self.pad_token: 0, self.unk_token: 1}
        self.idx2word = {0: self.pad_token, 1: self.unk_token}
        self.word_counts = {}
        
    def build_vocab(self, corpus):
        # Dem tan suat xuat hien cua tung tu trong toan bo tap van ban
        for text in corpus:
            tokens = text.split()
            for token in tokens:
                self.word_counts[token] = self.word_counts.get(token, 0) + 1
                
        # Them vao tu dien neu dat tan suat toi thieu
        for token, count in self.word_counts.items():
            if count >= self.min_freq and token not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[token] = idx
                self.idx2word[idx] = token
                
        print(f"Xay dung tu dien thanh cong! Tong so tu vung: {len(self.word2idx)}")
        
    def encode(self, text):
        # Chuyen cau van ban thanh danh sach chi so (Token IDs)
        tokens = text.split()
        return [self.word2idx.get(token, self.word2idx[self.unk_token]) for token in tokens]
        
    def decode(self, indices):
        # Chuyen nguoc lai tu chi so ve chuoi van ban
        return " ".join([self.idx2word.get(idx, self.unk_token) for idx in indices])

# Kiem thu Vocabulary tren tap ngu lieu nho
corpus = [
    clean_text("hoc may va hoc sau voi pytorch"),
    clean_text("olympic ai la dau truong dinh cao"),
    clean_text("hoc may giup giai quyet bai toan thuc te")
]

vocab = SimpleVocabulary(min_freq=1)
vocab.build_vocab(corpus)

test_sentence = clean_text("hoc may olympic robot") # 'robot' la tu moi chua tung gap (OOV)
encoded_ids = vocab.encode(test_sentence)
decoded_text = vocab.decode(encoded_ids)

print("\nCau kiem thu :", test_sentence)
print("Ma hoa thanh IDs:", encoded_ids)
print("Giai ma nguoc lai:", decoded_text)
print("-> Tu 'robot' khong co trong tu dien da duoc thay the an toan bang <UNK> (ID=1)!")

### 🔍 Code Walkthrough (Khối 2)
Khối code lập trình một lớp Vocabulary Indexer hoàn chỉnh từ số 0:
- Quản lý 2 chiều: `word2idx` (mã hóa văn bản thành số) và `idx2word` (giải mã số thành văn bản).
- Tự động thay thế các từ lạ bằng `<UNK>` (Index 1) mà không làm sập chương trình.
- Lưu trữ `<PAD>` ở vị trí 0 để sẵn sàng cho kỹ thuật padding batch sau này.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Lọc từ theo tần suất tối thiểu (min_freq)
Chạy lại phương thức `build_vocab` với `min_freq=2`. Quan sát xem từ điển còn lại bao nhiêu từ và giải thích tại sao trong các bài toán NLP lớn, việc loại bỏ các từ chỉ xuất hiện đúng 1 lần (typos/rác) giúp giảm mạnh kích thước mô hình.

### Bài 2 (Easy): Đếm số lượng từ lạ (OOV Rate)
Viết một hàm `calculate_oov_rate(text, vocab)` nhận vào một đoạn văn bản và trả về tỷ lệ phần trăm các từ bị chuyển thành `<UNK>`.

### Bài 3 (Medium): Xử lý từ dừng (Stopwords Removal)
Stopwords là các từ xuất hiện rất nhiều nhưng mang ít giá trị phân biệt thông tin (như: 'là', 'và', 'của', 'the', 'is', 'at'). Hãy tạo một danh sách các từ dừng đơn giản và viết hàm lọc bỏ chúng khỏi câu văn bản trước khi tokenize.

### Bài 4 (Medium): N-grams Tokenization
Tìm hiểu khái niệm n-grams (cặp 2 từ liên tiếp Bigram, 3 từ liên tiếp Trigram). Viết một hàm `extract_bigrams("hoc may va hoc sau")` trả về danh sách các cặp từ: `['hoc_may', 'may_va', 'va_hoc', 'hoc_sau']`. Nêu lợi ích của bigram trong việc giữ lại các cụm từ có nghĩa (ví dụ: 'không thích' khác hoàn toàn với 'thích').

### Bài 5 (Challenge): Mô phỏng thuật toán Byte-Pair Encoding (BPE)
Tìm hiểu nguyên lý của giải thuật Subword Tokenization BPE: Bắt đầu từ cấp độ ký tự, liên tục tìm cặp ký tự xuất hiện nhiều nhất và gộp chúng lại. Viết một đoạn mã ngắn mô phỏng một vòng lặp gộp ký tự của thuật toán này.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong các bài thi NLP của Olympic AI, việc xây dựng Từ điển (Vocabulary) là nơi cực kỳ dễ dính lỗi **Data Leakage**:
Nhiều thí sinh vô tình đưa cả tập `test.csv` vào để `build_vocab`. Điều này làm rò rỉ phân phối từ vựng của đề thi vào mô hình. Quy tắc chuẩn: **Chỉ học từ điển trên tập Train, các từ mới xuất hiện ở tập Test bắt buộc phải bị coi là `<UNK>`!** Hiểu sâu về Tokenization giúp bạn thiết lập ranh giới phòng vệ dữ liệu vững chắc.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Thí sinh mã hóa một câu văn bản nhưng quên xử lý từ vựng ngoài từ điển (OOV), dẫn tới lỗi `KeyError`:

```python
# CODE CÓ LỖI TRA CỨU TỪ ĐIỂN:
# word_to_id = {'ai': 0, 'olympic': 1}
# query = "thi dau ai"
# ids = [word_to_id[w] for w in query.split()] # 'thi' khong co trong dict -> KeyError: 'thi'
```

**Nhiệm vụ của bạn:**
1. Giải thích tại sao việc dùng phương thức `dict.get(key, default)` an toàn hơn nhiều so với toán tử truy xuất trực tiếp `dict[key]`.
2. Sửa lại code để luôn gán giá trị mặc định là mã của `<UNK>` khi không tìm thấy từ trong từ điển.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Lý do cơ bản nhất khiến máy tính không thể nạp trực tiếp chuỗi văn bản thô vào mạng nơ-ron là gì?
   - A. Mạng nơ-ron chỉ hoạt động trên các phép toán đại số ma trận số thực, không có cơ chế tính toán số học trực tiếp trên ký tự chuỗi.
   - B. Do file văn bản quá nhẹ.
   - C. Vì tiếng Việt có dấu.
   - D. Do giới hạn của hệ điều hành Windows.

2. Token đặc biệt `<UNK>` (Unknown) trong hệ thống xử lý ngôn ngữ tự nhiên đại diện cho điều gì?
   - A. Điểm kết thúc của câu văn bản.
   - B. Các từ lạ, từ viết sai chính tả hoặc từ chưa từng xuất hiện trong tập dữ liệu từ điển đã học (Out-Of-Vocabulary).
   - C. Khoảng trắng giữa các từ.
   - D. Dấu chấm câu.

3. Kỹ thuật Subword Tokenization (như BPE hay WordPiece) khắc phục được nhược điểm lớn nào của Word-level Tokenization truyền thống?
   - A. Tăng tốc độ quạt tản nhiệt của máy tính.
   - B. Giảm thiểu kích thước từ điển đồng thời giải quyết triệt để bài toán từ lạ (OOV) bằng cách bóc tách các từ mới thành các gốc từ và hậu tố quen thuộc.
   - C. Tự động dịch văn bản sang tiếng Anh.
   - D. Bỏ qua hoàn toàn bước huấn luyện.

4. Token đặc biệt `<PAD>` thường được gán giá trị chỉ số (Index) bằng bao nhiêu theo quy ước phổ biến nhất?
   - A. 9999
   - B. 0
   - C. -1
   - D. 255

5. Biểu thức chính quy (Regex) `re.sub(r'[^a-zA-Z0-9\s]', '', text)` thực hiện thao tác gì trên chuỗi?
   - A. Xóa toàn bộ chữ cái và chữ số.
   - B. Xóa tất cả các ký tự đặc biệt và dấu câu, chỉ giữ lại chữ cái, số và khoảng trắng.
   - C. Viết hoa toàn bộ văn bản.
   - D. Đếm số lượng từ trong câu.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Quy trình chuẩn hóa văn bản thô bằng Regular Expressions.
  - Các cấp độ Tokenization: Word, Character và Subword intuition.
  - Cấu trúc Vocabulary Indexer hai chiều và cơ chế xử lý từ lạ `<UNK>`.
* **What I can now do:**
  - Tự tay xây dựng một hệ thống Từ điển ngôn ngữ hoàn chỉnh từ đầu đến cuối.
  - Mã hóa an toàn bất kỳ câu văn bản nào thành danh sách Token IDs mà không lo lỗi sập code.
* **What I still need to practice:**
  - Khám phá các phương pháp trích xuất đặc trưng văn bản kinh điển: **Bag-of-Words & TF-IDF** ở Buổi 7.



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Trong **Buổi M4_S07**, chúng ta sẽ tiến lên một bước: **Trích Xuất Đặc Trưng Văn Bản: Bag-of-Words & TF-IDF với Scikit-Learn** — Tìm hiểu cách biến các văn bản thành ma trận vectơ tần suất từ vựng (Sparse Matrix) và huấn luyện mô hình phân loại cảm xúc văn bản đầu tiên!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M4**:
- Chuẩn hóa văn bản bằng Regex & hạ chữ thường ✓
- Tokenization (Word-level, Char-level, Subword intuition) ✓
- Xây dựng Vocabulary indexer & Xử lý từ lạ <UNK> ✓

* **Session Deliverable:** Mã nguồn Vocabulary Indexer chuẩn xử lý OOV an toàn
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
